# 03 — ADM: Diffusion Models Beat GANs on Image Synthesis

**Paper:** *Diffusion Models Beat GANs on Image Synthesis* (Dhariwal & Nichol, NeurIPS 2021)  
**arXiv:** https://arxiv.org/abs/2105.05233

---

## Motivation

By 2021, GANs (BigGAN, StyleGAN2) dominated class-conditional image generation.  
ADM (Architecture + Classifier guidance Diffusion Model) asks: can diffusion models beat them?

Answer: **Yes** — with two contributions:
1. **Better U-Net architecture** (ADM = Ablated Diffusion Model)
2. **Classifier guidance** — use a separately trained classifier to steer sampling

## Contribution 1: Improved U-Net Architecture

![ADM Architecture](./figures/adm_arch.png)

Key architectural changes over DDPM's U-Net:

| Component | DDPM | ADM |
|-----------|------|-----|
| Normalization | Group Norm | Group Norm (32 groups) |
| Attention | 16×16 resolution only | Multi-resolution (8, 16, 32) |
| Attention heads | Single | Multi-head |
| Upsampling | Nearest | Learned convolution |
| Residual scale | No | BigGAN-style residual scaling |
| Conditioning | Timestep only | Timestep + Class embedding (AdaGN) |

**AdaGN (Adaptive Group Normalization):**
```
AdaGN(h, y) = y_s · GroupNorm(h) + y_b
where [y_s, y_b] = Linear(timestep_embed + class_embed)
```
This is the diffusion equivalent of AdaIN in StyleGAN.

## Contribution 2: Classifier Guidance

![Classifier Guidance Results](./figures/adm_classifier_guidance.png)

The idea: use the **gradient of a classifier** to push samples towards a target class at inference time.

```
∇_{x_t} log p_φ(y | x_t)   ← gradient from auxiliary classifier trained on noisy images

Modified score:
    ε̃_θ(x_t, t, y) = ε_θ(x_t, t) − s · σ_t · ∇_{x_t} log p_φ(y | x_t)
    
    s = guidance scale  (s > 1 → stronger class adherence, less diversity)
```

This is analogous to Classifier-Free Guidance but uses a **real** classifier rather than a null conditioning.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ── AdaGN: Adaptive Group Normalization ──

class AdaGN(nn.Module):
    """
    Adaptive Group Normalization.
    Injects timestep + class conditioning into residual blocks.
    
    c: conditioning vector (B, d_cond)  = timestep_emb + class_emb
    """
    def __init__(self, num_channels, d_cond, num_groups=32):
        super().__init__()
        self.gn   = nn.GroupNorm(num_groups, num_channels, affine=False)
        self.proj = nn.Linear(d_cond, num_channels * 2)  # predict scale + shift
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x, c):
        # x: (B, C, H, W)   c: (B, d_cond)
        scale, shift = self.proj(c).chunk(2, dim=-1)
        scale = scale.view(-1, scale.shape[-1], 1, 1)
        shift = shift.view(-1, shift.shape[-1], 1, 1)
        return self.gn(x) * (1 + scale) + shift


# ── ADM Residual Block ──

class ADMResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, d_cond, dropout=0.1):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.adagn1 = AdaGN(out_ch, d_cond)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.adagn2 = AdaGN(out_ch, d_cond)
        self.drop   = nn.Dropout(dropout)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, c):
        h = F.silu(self.adagn1(self.conv1(x), c))
        h = self.drop(h)
        h = self.adagn2(self.conv2(h), c)
        return F.silu(h + self.skip(x))


# Test
block = ADMResBlock(64, 128, d_cond=256)
x = torch.randn(2, 64, 16, 16)
c = torch.randn(2, 256)
out = block(x, c)
print("ADMResBlock output:", out.shape)  # (2, 128, 16, 16)

In [ ]:
# ── Classifier Guidance: Gradient-based Steering ──

class NoisyClassifier(nn.Module):
    """
    Small classifier trained on *noisy* images to provide guidance gradients.
    Real ADM trains a full ResNet; here we show the principle.
    """
    def __init__(self, in_ch=4, d_model=64, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, d_model, 3, padding=1, stride=2),
            nn.SiLU(),
            nn.Conv2d(d_model, d_model*2, 3, padding=1, stride=2),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(d_model*2, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)


def classifier_guidance_step(eps_uncond, xt, t, classifier, y, s=7.5):
    """
    Modify noise prediction using classifier gradient.
    s: guidance scale
    """
    xt_req = xt.detach().requires_grad_(True)
    logits = classifier(xt_req)
    log_p  = F.log_softmax(logits, dim=-1)
    score  = log_p[range(len(y)), y].sum()
    grad   = torch.autograd.grad(score, xt_req)[0]

    # Sigma_t for scale (simplified: use sqrt(1-alpha_bar))
    sigma_t = 0.1  # would be schedule-dependent in practice
    eps_guided = eps_uncond - s * sigma_t * grad.detach()
    return eps_guided


# Demo
clf = NoisyClassifier(in_ch=4, num_classes=10)
xt  = torch.randn(2, 4, 16, 16)
t   = torch.tensor([500, 300])
y   = torch.tensor([3, 7])
eps = torch.randn_like(xt)

eps_guided = classifier_guidance_step(eps, xt, t, clf, y, s=7.5)
print(f"Original eps norm: {eps.norm():.3f}")
print(f"Guided  eps norm:  {eps_guided.norm():.3f}")

In [ ]:
# ── Guidance Scale vs FID / IS tradeoff ──

# Approximate FID/IS values from Figure 6 of the ADM paper
scales = [1.0, 2.5, 5.0, 7.5, 10.0, 15.0]
fid    = [27.5, 15.6, 10.9,  7.3,  6.0,  6.8]   # lower = better
is_    = [74,   118,  148,   186,  205,  215]     # higher = better

fig, ax1 = plt.subplots(figsize=(8, 4))
color1, color2 = 'steelblue', 'tomato'
ax2 = ax1.twinx()

ax1.plot(scales, fid, 'o-', color=color1, label='FID (↓)')
ax1.set_xlabel('Guidance Scale (s)', fontsize=12)
ax1.set_ylabel('FID-50K (↓)', color=color1, fontsize=11)
ax1.tick_params(axis='y', labelcolor=color1)

ax2.plot(scales, is_, 's--', color=color2, label='Inception Score (↑)')
ax2.set_ylabel('Inception Score (↑)', color=color2, fontsize=11)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.axvline(x=7.5, color='gray', linestyle=':', alpha=0.5, label='s=7.5 (paper default)')
ax1.set_title('Classifier Guidance Scale: FID vs Inception Score Tradeoff\n(ImageNet 256×256)', fontsize=12)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout(); plt.show()

print("Optimal guidance scale ≈ 7.5-10: best FID without diversity collapse")

## Results: Beating GANs

| Method | FID-50K (256×256) | IS |
|--------|-------------------|----|
| BigGAN-deep | 6.95 | 171.4 |
| StyleGAN-XL | 2.30 | — |
| **ADM (s=4.0)** | **10.94** | 100.98 |
| **ADM-G (s=7.5)** | **4.59** | 186.7 |
| **ADM-G + U** (upsampler) | **3.94** | 215.8 |

ADM-G (with guidance, s=7.5) surpasses BigGAN-deep on FID.  
The key insight: guidance **trades diversity for fidelity** — at `s > 1`, the model samples from a sharpened version of `p(x|y)^s`.

## Summary

| Component | Description |
|-----------|-------------|
| **Architecture** | U-Net with AdaGN, multi-scale attention, BigGAN residuals |
| **Classifier** | Separate model trained on noisy images, provides ∇ log p(y|x_t) |
| **Guidance formula** | ε̃ = ε_θ(x_t) − s·σ_t·∇ log p_φ(y\|x_t) |
| **Effect of s** | s=1 → no guidance; s>1 → trades diversity for fidelity |

### Key Insight

Classifier guidance provides a **test-time** mechanism to control the trade-off between sample quality and diversity — without retraining the diffusion model. This inspired Classifier-Free Guidance (used in DiT and all modern models).